# WebSocket + End-of-Session Summary

- WebSocket streaming chat
- REST `/chat/summary` khi đóng tab
- Đơn giản, dễ implement

In [12]:
import asyncio
import json
import requests
import websockets
from datetime import datetime

# ENDPOINTS
WS_URL = "ws://139.59.126.158:8000/ws/chat"
SUMMARY_URL = "http://139.59.126.158:8000/api/v1/chat/summary"

# CONFIG
MAX_HISTORY_TO_SEND = 20  # Chỉ gửi 20 messages gần nhất

---
## 1. Client State

In [13]:
# SESSION
thread_id = f"mvp-{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# SHORT-TERM (từ form + test)
user_profile = {
    "form_date": "2026-01-30",
    "basic_info": {
        "full_name": "Nguyễn Văn A",
        "age": 18,
        "address": "Hà Nội",
        "status": "hoc_sinh"
    }
}

holland_result = {
    "scores": {"R": 5, "I": 7, "A": 6, "S": 8, "E": 4, "C": 3},
    "top3": ["S", "I", "A"],
    "top_code": "SIA"
}

# LONG-TERM (load từ DB nếu có)
previous_summary = None  # Session mới = None

# HISTORY (session này)
history = []

print(f"Thread: {thread_id}")
print(f"User: {user_profile['basic_info']['full_name']}")
print(f"Previous Summary: {previous_summary}")

Thread: mvp-20260130_143332
User: Nguyễn Văn A
Previous Summary: None


---
## 2. WebSocket Chat

In [14]:
async def chat(message: str):
    """WebSocket streaming chat"""
    global history
    
    print(f"\n USER: {message}")
    
    # Payload - gửi 20 messages gần nhất + previous summary
    payload = {
        "message": message,
        "profile_json": user_profile,
        "test_json": holland_result,
        "history": history[-MAX_HISTORY_TO_SEND:],
        "session_summary": previous_summary
    }
    
    full_response = ""
    
    async with websockets.connect(f"{WS_URL}/{thread_id}") as ws:
        await ws.send(json.dumps(payload))
        print(" AI: ", end="", flush=True)
        
        async for msg in ws:
            data = json.loads(msg)
            
            if data.get("type") == "token":
                print(data.get("content", ""), end="", flush=True)
                full_response += data.get("content", "")
            
            elif data.get("type") == "end":
                print("\n")
                break
            
            elif data.get("type") == "error":
                print(f"\n {data.get('content')}")
                return
    
    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": full_response})
    
    print(f" History: {len(history)} messages")

---
## 3. Demo Chat

In [15]:
questions = [
    "Chào bạn, em cần tư vấn ngành học",
    "Em thích toán và lập trình",
    "Điểm chuẩn CNTT?",
    "Cảm ơn bạn!"
]

async def demo():
    for q in questions:
        await chat(q)
        await asyncio.sleep(0.3)

await demo()


 USER: Chào bạn, em cần tư vấn ngành học
 AI: Chào bạn Nguyễn Văn A! Mình rất vui được giúp bạn chọn ngành học phù hợp. 

Trước tiên, bạn cho mình biết chút nhé:
1) Bạn có thích hoặc quan tâm đến lĩnh vực nào không?
2) Môn học nào bạn cảm thấy học tốt hoặc thích nhất?
3) Bạn đã làm trắc nghiệm Holland (RIASEC) rồi, kết quả của bạn là SIA, mình sẽ giải thích và gợi ý nghề phù hợp cho bạn nhé.

Bạn muốn bắt đầu từ đâu?

 History: 2 messages

 USER: Em thích toán và lập trình
 AI: Ok, bạn thích toán và lập trình thì rất hợp với nhóm Investigative (I) trong kết quả Holland của bạn rồi. Kết quả SIA của bạn có:

- S (Social): Thích giúp đỡ, giao tiếp, làm việc với người
- I (Investigative): Thích phân tích, nghiên cứu, giải quyết vấn đề (rất phù hợp với toán và lập trình)
- A (Artistic): Có xu hướng sáng tạo, độc lập

Với sở thích toán và lập trình cùng tính cách SIA, mình gợi ý bạn có thể cân nhắc các ngành như:

- Công nghệ thông tin (CNTT), Khoa học máy tính
- Kỹ thuật phần mềm
- An toàn

---
## 4. End-of-Session Summary

**Gọi khi**: User đóng tab / logout / session timeout

In [16]:
def save_session_summary():
    """Gọi khi đóng tab - Lưu summary vào DB"""
    
    print(" END OF SESSION")
    print(f"   History: {len(history)} messages")
    
    # Gọi API
    response = requests.post(SUMMARY_URL, json={
        "thread_id": thread_id,
        "history": history[-MAX_HISTORY_TO_SEND:],
        "existing_summary": previous_summary
    })
    
    result = response.json()
    
    print(f"\n SUMMARY TO SAVE:")
    print(f"   {result.get('summary', 'N/A')}")
    print(f"\n Backend lưu vào DB với thread_id = {thread_id}")
    
    return result

# Simulate đóng tab
save_session_summary()

 END OF SESSION
   History: 8 messages

 SUMMARY TO SAVE:
   The user, Nguyễn Văn A, seeks guidance on choosing a suitable academic major. He has a strong interest in mathematics and programming, aligning well with the Investigative (I) personality type from the Holland code assessment. Additionally, his SIA profile indicates he enjoys social interaction and helping others. He inquired about the admission scores for Information Technology programs in 2024, showing interest in specific universities such as Hanoi University of Science and Technology and University of Engineering and Technology (Vietnam National University). The conversation focused on matching his strengths and preferences with potential study fields, particularly IT. The user expressed gratitude and openness to further advice. Future sessions should consider his affinity for math, programming, and social engagement when recommending academic or career paths.

 Backend lưu vào DB với thread_id = mvp-20260130_143332


{'thread_id': 'mvp-20260130_143332',
 'summary': 'The user, Nguyễn Văn A, seeks guidance on choosing a suitable academic major. He has a strong interest in mathematics and programming, aligning well with the Investigative (I) personality type from the Holland code assessment. Additionally, his SIA profile indicates he enjoys social interaction and helping others. He inquired about the admission scores for Information Technology programs in 2024, showing interest in specific universities such as Hanoi University of Science and Technology and University of Engineering and Technology (Vietnam National University). The conversation focused on matching his strengths and preferences with potential study fields, particularly IT. The user expressed gratitude and openness to further advice. Future sessions should consider his affinity for math, programming, and social engagement when recommending academic or career paths.',
 'message_count': 8,
 'timestamp': '2026-01-30T07:33:51.188105'}

---
## Backend Flow

```
┌─────────────────────────────────────────────────────────────┐
│                     MVP FLOW                                 │
└─────────────────────────────────────────────────────────────┘

1. USER MỞ CHAT
   └─→ Load previous_summary từ DB (nếu có)

2. MỖI LẦN CHAT (WebSocket)
   └─→ Gửi: message + profile + holland + history[20] + summary
   └─→ Nhận: streaming response
   └─→ Append to history (local)

3. USER ĐÓNG TAB (REST)
   └─→ POST /chat/summary
   └─→ Backend tạo summary từ history
   └─→ Lưu summary vào DB

4. USER QUAY LẠI
   └─→ Load summary từ DB
   └─→ Bắt đầu session mới với context từ summary
```